In [ ]:
import json
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score

In [ ]:
train_data = json.load(open('./data.json'))
train_data[:2]

In [ ]:
df = pd.DataFrame(train_data)
print(df["label"].value_counts())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print("train:", len(X_train))
print("test :", len(X_test))

In [ ]:
router = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=1,
            max_df=0.95,
            sublinear_tf=True
        )
    ),
    (
        "clf",
        LogisticRegression(
            max_iter=2000,
            random_state=42
        )
    )
])

In [ ]:
router.fit(X_train, y_train)

In [ ]:
y_pred = router.predict(X_test)

print("accuracy:", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=router.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=router.classes_)
disp.plot(cmap="Blues")
plt.show()

In [ ]:
joblib.dump(router, "router_tfidf.pkl")
print("saved: router_tfidf.pkl")